# Bike Sharing Demand Prediction — End-to-End Regression Project
**Two separate modeling tracks:** daily demand (`day.csv`) and hourly demand (`hour.csv`)

This notebook is designed as a reusable project workflow: business understanding, data validation, leakage checks, exploratory analysis, preprocessing, multiple regression models, regularization, evaluation, residual diagnostics, overfitting checks, and final model selection.

### Business objective
Predict the total number of bike rentals (`cnt`) using information available at prediction time, such as calendar, season, weather, and (for the hourly dataset) hour of day.

### Important modeling decisions
- **Target:** `cnt` (total rentals).
- **Leakage prevention:** exclude `casual` and `registered`, because `cnt = casual + registered`. These columns reveal the target mathematically and must not be used as predictors.
- **Exclude `instant`:** it is a row identifier, not a meaningful demand driver.
- **Date-aware split:** preserve chronological order and use the final 20% of dates as a holdout set. This is more realistic than randomly mixing past and future observations.
- **Separate datasets:** daily and hourly rows represent different prediction horizons; evaluate and select models independently.
- **Categorical features:** numeric codes such as season, month, weekday, weather situation, and hour are categories, not necessarily continuous measurements.
- **Normality:** residual normality is a diagnostic for classical linear-model inference, not a requirement for accurate prediction. Inspect it alongside residual patterns and error metrics.

## 1. Setup and imports

Place `day.csv` and `hour.csv` in the same folder as this notebook, or change the file paths in the next cell. The code accepts common date formats such as `01-01-2011` and `2011-01-01`.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit, cross_validate, learning_curve
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor

RANDOM_STATE = 42
DATA_DIR = Path(".")  # Change this if your CSV files are stored elsewhere
DAY_PATH = DATA_DIR / "day.csv"
HOUR_PATH = DATA_DIR / "hour.csv"

print("Working directory:", Path.cwd())
print("day.csv found:", DAY_PATH.exists())
print("hour.csv found:", HOUR_PATH.exists())

## 2. Load the two datasets

If a file is not found, upload it to the notebook's working directory or update `DATA_DIR`. The sample rows in the project brief are not sufficient for fitting a model; the full CSV files are required.

In [ ]:
def load_bike_data(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path.resolve()}. Put the CSV in the notebook folder "
            "or update DATA_DIR / DAY_PATH / HOUR_PATH."
        )
    df = pd.read_csv(path)
    # Normalize accidental spaces in column names.
    df.columns = df.columns.str.strip()
    return df

day_raw = load_bike_data(DAY_PATH)
hour_raw = load_bike_data(HOUR_PATH)

print("Daily data:", day_raw.shape)
print("Hourly data:", hour_raw.shape)
display(day_raw.head())
display(hour_raw.head())

## 3. Understand the columns and validate the data

The target `cnt` is total rental demand. Weather measurements (`temp`, `atemp`, `hum`, `windspeed`) in this commonly used dataset are scaled values, not necessarily raw Celsius / percentage / km/h. Do not interpret them as raw units without the dataset's data dictionary.

We check missing values, duplicate rows, data types, target validity, and the identity `cnt = casual + registered`.

In [ ]:
def quality_report(df, name):
    print(f"--- {name} ---")
    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("\nData types:")
    display(df.dtypes.rename("dtype").to_frame())
    print("\nMissing values:")
    display(df.isna().sum().sort_values(ascending=False).rename("missing").to_frame())
    if "cnt" in df.columns:
        print("\nTarget summary:")
        display(df["cnt"].describe().to_frame())
        print("Negative target values:", (df["cnt"] < 0).sum())
    if {"cnt", "casual", "registered"}.issubset(df.columns):
        mismatch = (df["cnt"] != df["casual"] + df["registered"]).sum()
        print(f"Rows where cnt != casual + registered: {mismatch}")

quality_report(day_raw, "Daily dataset")
quality_report(hour_raw, "Hourly dataset")

## 4. Exploratory data analysis (EDA)

Start with the target distribution, demand over time, and the relationship between demand and key numeric weather variables. EDA is used to understand the problem and identify patterns; it should not be used to make decisions based on the holdout target results.

In [ ]:
def parse_date_column(df):
    df = df.copy()
    if "dteday" in df.columns:
        # dayfirst=True handles the dd-mm-yyyy sample format.
        parsed = pd.to_datetime(df["dteday"], dayfirst=True, errors="coerce")
        # If parsing failed substantially, try the alternative convention.
        if parsed.isna().mean() > 0.25:
            parsed_alt = pd.to_datetime(df["dteday"], errors="coerce")
            if parsed_alt.isna().mean() < parsed.isna().mean():
                parsed = parsed_alt
        df["dteday"] = parsed
    return df

day_eda = parse_date_column(day_raw)
hour_eda = parse_date_column(hour_raw)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(day_eda["cnt"].dropna(), bins=30, edgecolor="black")
axes[0].set_title("Daily rental count distribution")
axes[0].set_xlabel("Total rentals (cnt)")
axes[0].set_ylabel("Number of days")
axes[1].hist(hour_eda["cnt"].dropna(), bins=40, edgecolor="black")
axes[1].set_title("Hourly rental count distribution")
axes[1].set_xlabel("Total rentals (cnt)")
axes[1].set_ylabel("Number of hours")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
axes[0].plot(day_eda["dteday"], day_eda["cnt"], linewidth=1)
axes[0].set_title("Daily rentals over time")
axes[0].set_ylabel("cnt")
axes[1].plot(hour_eda["dteday"], hour_eda["cnt"], linewidth=0.7)
axes[1].set_title("Hourly rentals over time (date-level axis)")
axes[1].set_ylabel("cnt")
plt.tight_layout()
plt.show()

In [ ]:
def plot_target_by_category(df, col, title):
    if col not in df.columns:
        return
    summary = df.groupby(col, dropna=False)["cnt"].mean().sort_index()
    plt.figure(figsize=(9, 4))
    plt.bar(summary.index.astype(str), summary.values, edgecolor="black")
    plt.title(title)
    plt.xlabel(col)
    plt.ylabel("Average rentals (cnt)")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

for col in ["season", "mnth", "weekday", "weathersit", "workingday"]:
    plot_target_by_category(day_eda, col, f"Daily average rentals by {col}")

plot_target_by_category(hour_eda, "hr", "Hourly average rentals by hour of day")
plot_target_by_category(hour_eda, "weathersit", "Hourly average rentals by weather situation")

weather_cols = [c for c in ["temp", "atemp", "hum", "windspeed"] if c in day_eda.columns]
day_eda[weather_cols + ["cnt"]].hist(figsize=(12, 7), bins=25, edgecolor="black")
plt.suptitle("Daily weather features and target distributions")
plt.tight_layout()
plt.show()

## 5. Feature engineering and leakage prevention

### Columns excluded from predictors
- `cnt`: target itself.
- `casual`, `registered`: components that sum to `cnt`; including them would create severe target leakage.
- `instant`: sequential record ID that may encourage memorizing the dataset rather than learning useful demand drivers.
- `dteday`: the raw date string is replaced with calendar-derived numeric features.

### Calendar features
We extract year, day of month, and day of year. Month, weekday, season, weather situation, and hour are encoded as categorical variables. Cyclic sine/cosine features represent the wrap-around nature of hour, weekday, month, and day of year. These are useful for linear models because, for example, 23:00 is close to 00:00 rather than far away numerically.

In [ ]:
def prepare_features(raw_df, dataset_kind):
    df = parse_date_column(raw_df)
    if "dteday" not in df.columns:
        raise ValueError("Expected a 'dteday' date column.")
    if df["dteday"].isna().any():
        bad = int(df["dteday"].isna().sum())
        raise ValueError(f"{bad} dates could not be parsed. Check the dteday format before modeling.")
    if "cnt" not in df.columns:
        raise ValueError("Expected target column 'cnt'.")

    df = df.sort_values("dteday").reset_index(drop=True)
    df["year_from_start"] = (df["dteday"] - df["dteday"].min()).dt.days / 365.25
    df["day_of_month"] = df["dteday"].dt.day
    df["day_of_year"] = df["dteday"].dt.dayofyear

    # Cyclic representations preserve wrap-around closeness.
    df["month_sin"] = np.sin(2 * np.pi * df["dteday"].dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["dteday"].dt.month / 12)
    df["weekday_sin"] = np.sin(2 * np.pi * df["dteday"].dt.dayofweek / 7)
    df["weekday_cos"] = np.cos(2 * np.pi * df["dteday"].dt.dayofweek / 7)
    df["doy_sin"] = np.sin(2 * np.pi * df["dteday"].dt.dayofyear / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * df["dteday"].dt.dayofyear / 365.25)

    if dataset_kind == "hour":
        if "hr" not in df.columns:
            raise ValueError("Hourly dataset should contain the 'hr' column.")
        df["hour_sin"] = np.sin(2 * np.pi * df["hr"] / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df["hr"] / 24)

    # The date itself is not passed as raw text to the model.
    excluded = {"cnt", "casual", "registered", "instant", "dteday"}
    X = df.drop(columns=[c for c in excluded if c in df.columns]).copy()
    y = df["cnt"].astype(float).copy()
    dates = df["dteday"].copy()

    # Known numeric-coded categories: treat as categories, not continuous scales.
    categorical = [c for c in [
        "season", "yr", "mnth", "holiday", "weekday", "workingday",
        "weathersit", "hr", "day_of_month"
    ] if c in X.columns]
    # Numeric features remaining after categorical selection.
    numeric = [c for c in X.columns if c not in categorical]

    # Convert categories to strings so OneHotEncoder treats them consistently.
    for c in categorical:
        X[c] = X[c].astype("Int64").astype(str)

    return X, y, dates, categorical, numeric, df

day_X, day_y, day_dates, day_cat, day_num, day_model_data = prepare_features(day_raw, "day")
hour_X, hour_y, hour_dates, hour_cat, hour_num, hour_model_data = prepare_features(hour_raw, "hour")

print("Daily predictors:", day_X.columns.tolist())
print("Hourly predictors:", hour_X.columns.tolist())
print("\nExcluded leakage / ID fields: cnt, casual, registered, instant, dteday")

## 6. Chronological train / test split

We hold out the last 20% of rows after sorting by date. The holdout is not used for model selection or tuning. For hourly data, rows from the same date can be split between train and test if the cutoff occurs within a day; this is still chronological, but for a stricter deployment simulation you can change the split to a whole-date boundary. The cell below uses the date cutoff to keep complete dates together.

In [ ]:
def chronological_split(X, y, dates, test_fraction=0.20):
    unique_dates = np.array(sorted(pd.Series(dates).dropna().unique()))
    if len(unique_dates) < 5:
        raise ValueError("Not enough distinct dates for a chronological holdout.")
    cutoff_idx = max(1, int(np.floor(len(unique_dates) * (1 - test_fraction))))
    cutoff_idx = min(cutoff_idx, len(unique_dates) - 1)
    cutoff_date = pd.Timestamp(unique_dates[cutoff_idx])

    train_mask = pd.Series(dates).values < np.datetime64(cutoff_date)
    test_mask = ~train_mask
    return (
        X.loc[train_mask].reset_index(drop=True), X.loc[test_mask].reset_index(drop=True),
        y.loc[train_mask].reset_index(drop=True), y.loc[test_mask].reset_index(drop=True),
        pd.Series(dates).loc[train_mask].reset_index(drop=True),
        pd.Series(dates).loc[test_mask].reset_index(drop=True),
        cutoff_date
    )

day_X_train, day_X_test, day_y_train, day_y_test, day_train_dates, day_test_dates, day_cutoff = chronological_split(day_X, day_y, day_dates)
hour_X_train, hour_X_test, hour_y_train, hour_y_test, hour_train_dates, hour_test_dates, hour_cutoff = chronological_split(hour_X, hour_y, hour_dates)

print(f"Daily cutoff date: {day_cutoff.date()}")
print(f"Daily train: {day_X_train.shape}, test: {day_X_test.shape}")
print(f"Hourly cutoff date: {hour_cutoff.date()}")
print(f"Hourly train: {hour_X_train.shape}, test: {hour_X_test.shape}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(day_train_dates, bins=20, alpha=0.8, label="Train")
axes[0].hist(day_test_dates, bins=20, alpha=0.8, label="Test")
axes[0].set_title("Daily train/test date coverage")
axes[0].legend()
axes[1].hist(hour_train_dates, bins=20, alpha=0.8, label="Train")
axes[1].hist(hour_test_dates, bins=20, alpha=0.8, label="Test")
axes[1].set_title("Hourly train/test date coverage")
axes[1].legend()
plt.tight_layout()
plt.show()

## 7. Preprocessing pipeline

Numeric features are median-imputed and standardized. Categorical features are most-frequent-imputed and one-hot encoded. Scaling matters for Linear Regression with regularization (Ridge, Lasso, Elastic Net); tree models do not require scaling, but using a consistent pipeline keeps the workflow easy to compare.

The preprocessing is fitted **only on training data** through the scikit-learn `Pipeline`, preventing information from the holdout set leaking into preprocessing.

In [ ]:
def make_onehot_encoder():
    # Dense output works with HistGradientBoostingRegressor; support older sklearn versions too.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_preprocessor(categorical, numeric):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder())
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric),
        ("categorical", categorical_pipe, categorical)
    ], remainder="drop")

day_preprocessor = make_preprocessor(day_cat, day_num)
hour_preprocessor = make_preprocessor(hour_cat, hour_num)

print("Daily numeric features:", day_num)
print("Daily categorical features:", day_cat)
print("Hourly numeric features:", hour_num)
print("Hourly categorical features:", hour_cat)

## 8. Train and compare multiple regression models

We compare:
- **Dummy Regressor:** baseline that predicts the training mean.
- **Linear Regression:** interpretable linear baseline.
- **Ridge:** L2 regularization; shrinks coefficients to reduce variance.
- **Lasso:** L1 regularization; can shrink some coefficients to zero.
- **Elastic Net:** combines L1 and L2 penalties.
- **Decision Tree, Random Forest, Extra Trees, Gradient Boosting, HistGradientBoosting:** nonlinear models that can capture interactions and thresholds.

Metrics:
- **MAE:** average absolute error in rentals; easier to interpret.
- **RMSE:** penalizes large errors more heavily.
- **R²:** proportion of target variance explained relative to a mean baseline; can be negative on a difficult holdout.
- **Train vs test metrics:** a large gap may indicate overfitting, but compare metrics in the context of time drift and model complexity.

In [ ]:
def build_models(preprocessor):
    estimators = {
        "Dummy mean baseline": DummyRegressor(strategy="mean"),
        "Linear Regression": LinearRegression(),
        "Ridge (alpha=10)": Ridge(alpha=10.0),
        "Lasso (alpha=0.01)": Lasso(alpha=0.01, max_iter=20000),
        "Elastic Net": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=20000),
        "Decision Tree": DecisionTreeRegressor(max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE),
        "Random Forest": RandomForestRegressor(n_estimators=250, min_samples_leaf=2, max_features=0.9, n_jobs=-1, random_state=RANDOM_STATE),
        "Extra Trees": ExtraTreesRegressor(n_estimators=250, min_samples_leaf=2, max_features=0.9, n_jobs=-1, random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=180, learning_rate=0.05, max_depth=2, loss="squared_error", random_state=RANDOM_STATE),
        "Hist Gradient Boosting": HistGradientBoostingRegressor(max_iter=200, learning_rate=0.06, l2_regularization=1.0, random_state=RANDOM_STATE)
    }
    return {
        name: Pipeline([("preprocess", clone(preprocessor)), ("model", estimator)])
        for name, estimator in estimators.items()
    }

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred)
    }

def fit_compare_models(X_train, X_test, y_train, y_test, preprocessor):
    models = build_models(preprocessor)
    rows = []
    fitted = {}
    predictions = {}
    for name, pipe in models.items():
        pipe.fit(X_train, y_train)
        train_pred = pipe.predict(X_train)
        test_pred = pipe.predict(X_test)
        train_m = regression_metrics(y_train, train_pred)
        test_m = regression_metrics(y_test, test_pred)
        rows.append({
            "Model": name,
            "Train MAE": train_m["MAE"],
            "Test MAE": test_m["MAE"],
            "Train RMSE": train_m["RMSE"],
            "Test RMSE": test_m["RMSE"],
            "Train R2": train_m["R2"],
            "Test R2": test_m["R2"],
            "RMSE gap (test-train)": test_m["RMSE"] - train_m["RMSE"]
        })
        fitted[name] = pipe
        predictions[name] = test_pred
    results = pd.DataFrame(rows).sort_values("Test RMSE").reset_index(drop=True)
    return results, fitted, predictions

day_results, day_models, day_predictions = fit_compare_models(
    day_X_train, day_X_test, day_y_train, day_y_test, day_preprocessor
)
hour_results, hour_models, hour_predictions = fit_compare_models(
    hour_X_train, hour_X_test, hour_y_train, hour_y_test, hour_preprocessor
)

print("DAILY MODEL COMPARISON")
display(day_results.round(3))
print("HOURLY MODEL COMPARISON")
display(hour_results.round(3))

In [ ]:
def plot_model_comparison(results, title):
    plot_df = results.sort_values("Test RMSE", ascending=True)
    plt.figure(figsize=(11, 5))
    plt.barh(plot_df["Model"], plot_df["Test RMSE"], edgecolor="black")
    plt.gca().invert_yaxis()
    plt.title(title)
    plt.xlabel("Holdout RMSE (lower is better)")
    plt.tight_layout()
    plt.show()

plot_model_comparison(day_results, "Daily demand: holdout RMSE by model")
plot_model_comparison(hour_results, "Hourly demand: holdout RMSE by model")

## 9. Focused comparison: Linear, Ridge, Lasso, and Elastic Net

Regularization penalizes large coefficients. The penalty strength `alpha` controls how strongly coefficients are constrained. Standardizing numeric features is important because penalty terms are scale-sensitive. One-hot encoded categories are included in the same regularized model.

This section uses a small alpha grid and reports holdout performance. **Do not repeatedly tune against the final test set in a production project**; use a validation period or time-series cross-validation for tuning, then evaluate the selected configuration once on the final holdout.

In [ ]:
def regularization_sweep(X_train, X_test, y_train, y_test, preprocessor):
    rows = []
    fitted = {}
    for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
        candidates = {
            f"Ridge alpha={alpha}": Ridge(alpha=alpha),
            f"Lasso alpha={alpha}": Lasso(alpha=alpha, max_iter=30000),
            f"ElasticNet alpha={alpha}": ElasticNet(alpha=alpha, l1_ratio=0.5, max_iter=30000)
        }
        for name, estimator in candidates.items():
            pipe = Pipeline([("preprocess", clone(preprocessor)), ("model", estimator)])
            pipe.fit(X_train, y_train)
            pred_train = pipe.predict(X_train)
            pred_test = pipe.predict(X_test)
            mt = regression_metrics(y_train, pred_train)
            mv = regression_metrics(y_test, pred_test)
            rows.append({
                "Model": name, "Train RMSE": mt["RMSE"], "Test RMSE": mv["RMSE"],
                "Test MAE": mv["MAE"], "Test R2": mv["R2"],
                "RMSE gap": mv["RMSE"] - mt["RMSE"]
            })
            fitted[name] = pipe
    return pd.DataFrame(rows).sort_values("Test RMSE").reset_index(drop=True), fitted

day_reg_results, day_reg_models = regularization_sweep(day_X_train, day_X_test, day_y_train, day_y_test, day_preprocessor)
hour_reg_results, hour_reg_models = regularization_sweep(hour_X_train, hour_X_test, hour_y_train, hour_y_test, hour_preprocessor)

print("Daily regularization sweep")
display(day_reg_results.head(10).round(3))
print("Hourly regularization sweep")
display(hour_reg_results.head(10).round(3))

## 10. Select a candidate and inspect holdout predictions

The default selection rule below picks the lowest holdout RMSE. In a formal workflow, select the model using time-aware validation on the training period, then use the holdout only for final reporting. Here the holdout ranking is shown as an instructional comparison.

In [ ]:
day_best_name = day_results.iloc[0]["Model"]
hour_best_name = hour_results.iloc[0]["Model"]
day_best_model = day_models[day_best_name]
hour_best_model = hour_models[hour_best_name]
day_best_pred = day_predictions[day_best_name]
hour_best_pred = hour_predictions[hour_best_name]

print("Best daily candidate by holdout RMSE:", day_best_name)
print("Best hourly candidate by holdout RMSE:", hour_best_name)

day_prediction_table = pd.DataFrame({
    "date": day_test_dates,
    "actual_cnt": day_y_test,
    "predicted_cnt": day_best_pred,
})
day_prediction_table["residual_actual_minus_predicted"] = day_prediction_table["actual_cnt"] - day_prediction_table["predicted_cnt"]

hour_prediction_table = pd.DataFrame({
    "date": hour_test_dates,
    "actual_cnt": hour_y_test,
    "predicted_cnt": hour_best_pred,
})
hour_prediction_table["residual_actual_minus_predicted"] = hour_prediction_table["actual_cnt"] - hour_prediction_table["predicted_cnt"]

display(day_prediction_table.head(15).round(2))
display(hour_prediction_table.head(15).round(2))

## 11. Required regression visual diagnostics

For each selected candidate, inspect:
1. **Actual vs predicted:** points close to the diagonal indicate good predictions.
2. **Residuals vs fitted:** a curve or funnel can signal nonlinearity or non-constant error variance.
3. **Residuals over time:** drift suggests that demand patterns changed over time.
4. **Residual histogram and Q–Q plot:** check whether residuals are roughly symmetric / normal. Perfect normality is not expected.
5. **Error by actual demand band:** shows whether the model performs worse during peak-demand periods.

Residual definition used here: `actual - predicted`. Positive residual means the model underpredicted demand.

In [ ]:
def diagnostic_plots(y_true, y_pred, dates, title):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    residuals = y_true - y_pred

    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    axes[0, 0].scatter(y_true, y_pred, alpha=0.55)
    low = min(y_true.min(), y_pred.min())
    high = max(y_true.max(), y_pred.max())
    axes[0, 0].plot([low, high], [low, high], linestyle="--")
    axes[0, 0].set_title("Actual vs predicted")
    axes[0, 0].set_xlabel("Actual rentals")
    axes[0, 0].set_ylabel("Predicted rentals")

    axes[0, 1].scatter(y_pred, residuals, alpha=0.55)
    axes[0, 1].axhline(0, linestyle="--")
    axes[0, 1].set_title("Residuals vs fitted")
    axes[0, 1].set_xlabel("Predicted rentals")
    axes[0, 1].set_ylabel("Actual - predicted")

    axes[1, 0].hist(residuals, bins=35, edgecolor="black")
    axes[1, 0].axvline(0, linestyle="--")
    axes[1, 0].set_title("Residual distribution")
    axes[1, 0].set_xlabel("Residual")

    # Q-Q plot with scipy when available; fallback to sorted residuals.
    axes[1, 1].set_title("Residual Q-Q plot")
    try:
        from scipy import stats
        stats.probplot(residuals, dist="norm", plot=axes[1, 1])
    except Exception:
        sorted_res = np.sort(residuals)
        theoretical = np.linspace(-2.5, 2.5, len(sorted_res))
        axes[1, 1].scatter(theoretical, sorted_res, s=12)
        axes[1, 1].set_xlabel("Approximate theoretical quantiles")
        axes[1, 1].set_ylabel("Observed residual quantiles")

    fig.suptitle(title, y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(13, 4))
    plt.plot(pd.to_datetime(dates), residuals, linewidth=0.8)
    plt.axhline(0, linestyle="--")
    plt.title(f"{title}: residuals over time")
    plt.xlabel("Date")
    plt.ylabel("Actual - predicted")
    plt.tight_layout()
    plt.show()

    # Compare error across demand levels.
    bands = pd.qcut(pd.Series(y_true), q=min(5, len(np.unique(y_true))), duplicates="drop")
    error_by_band = pd.DataFrame({"actual": y_true, "absolute_error": np.abs(residuals), "band": bands})
    summary = error_by_band.groupby("band", observed=False).agg(
        rows=("absolute_error", "size"),
        mean_absolute_error=("absolute_error", "mean"),
        mean_actual_demand=("actual", "mean")
    )
    display(summary.round(2))

diagnostic_plots(day_y_test, day_best_pred, day_test_dates, f"Daily — {day_best_name}")
diagnostic_plots(hour_y_test, hour_best_pred, hour_test_dates, f"Hourly — {hour_best_name}")

## 12. Check for overfitting

A model may be overfitting when training error is very low but holdout error is much higher. The table's train/test metrics provide an initial check. A large gap is a warning, not proof by itself: chronological holdout data may differ from the training period.

Possible remedies:
- constrain tree depth and increase minimum leaf size;
- increase Ridge / Lasso / Elastic Net regularization;
- reduce unnecessary features;
- use time-aware cross-validation;
- gather more representative data and investigate drift;
- check whether the prediction scenario uses only features available at prediction time.

In [ ]:
def overfit_report(results, title):
    view = results.copy()
    view["RMSE ratio (test/train)"] = view["Test RMSE"] / view["Train RMSE"].replace(0, np.nan)
    view["Heuristic flag"] = np.where(
        view["RMSE ratio (test/train)"] >= 1.5,
        "Review: large train/test gap",
        "No large gap by this rule"
    )
    print(title)
    display(view.sort_values("Test RMSE")[[
        "Model", "Train RMSE", "Test RMSE", "RMSE gap (test-train)",
        "RMSE ratio (test/train)", "Heuristic flag"
    ]].round(3))

overfit_report(day_results, "Daily overfitting review")
overfit_report(hour_results, "Hourly overfitting review")

## 13. Learning curves for a closer generalization check

Learning curves show training and validation error as the training sample grows. The validation strategy here uses expanding time splits (`TimeSeriesSplit`) rather than random folds. Lower validation error with more data can suggest that additional representative training data would help. A persistent gap between training and validation error can indicate high variance / overfitting.

In [ ]:
def plot_learning_curve_time_series(estimator, X, y, title):
    cv = TimeSeriesSplit(n_splits=4)
    sizes, train_scores, val_scores = learning_curve(
        estimator, X, y,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        train_sizes=np.linspace(0.3, 1.0, 5),
        n_jobs=-1,
        shuffle=False
    )
    train_rmse = -train_scores.mean(axis=1)
    val_rmse = -val_scores.mean(axis=1)
    plt.figure(figsize=(8, 5))
    plt.plot(sizes, train_rmse, marker="o", label="Training RMSE")
    plt.plot(sizes, val_rmse, marker="o", label="Time-split validation RMSE")
    plt.title(title)
    plt.xlabel("Number of training rows")
    plt.ylabel("RMSE (lower is better)")
    plt.legend()
    plt.tight_layout()
    plt.show()

# These cells may take a little time, especially for ensemble models.
plot_learning_curve_time_series(day_best_model, day_X_train, day_y_train, f"Daily learning curve — {day_best_name}")
plot_learning_curve_time_series(hour_best_model, hour_X_train, hour_y_train, f"Hourly learning curve — {hour_best_name}")

## 14. Coefficient interpretation for linear and regularized models

Coefficients are interpretable only relative to the preprocessing pipeline. Numeric features are standardized, and categorical features are one-hot encoded. A positive coefficient indicates a higher predicted rental count relative to the model's reference/baseline conditions, holding other encoded features constant. Correlated features can make individual coefficients unstable.

This section extracts coefficients for the linear-family models when supported. It does not imply causation.

In [ ]:
def coefficient_table(pipe, top_n=20):
    model = pipe.named_steps["model"]
    if not hasattr(model, "coef_"):
        print("This model does not expose linear coefficients.")
        return None
    pre = pipe.named_steps["preprocess"]
    try:
        names = pre.get_feature_names_out()
    except Exception:
        names = [f"feature_{i}" for i in range(len(model.coef_))]
    coefs = np.asarray(model.coef_).ravel()
    table = pd.DataFrame({"feature": names, "coefficient": coefs})
    table["absolute_coefficient"] = table["coefficient"].abs()
    return table.sort_values("absolute_coefficient", ascending=False).head(top_n)

for model_name in ["Linear Regression", "Ridge (alpha=10)", "Lasso (alpha=0.01)", "Elastic Net"]:
    if model_name in day_models:
        print("Daily:", model_name)
        display(coefficient_table(day_models[model_name], top_n=15))
    if model_name in hour_models:
        print("Hourly:", model_name)
        display(coefficient_table(hour_models[model_name], top_n=15))

## 15. Time-aware cross-validation on the training period

The final holdout should remain untouched while choosing a model. This section demonstrates expanding-window validation on the training data for a shortlist of models. Scores are not directly interchangeable with the final holdout score because each fold tests on a different future period.

In [ ]:
def time_cv_shortlist(X_train, y_train, preprocessor, dataset_name):
    shortlist = {
        "Linear Regression": LinearRegression(),
        "Ridge (alpha=10)": Ridge(alpha=10.0),
        "Lasso (alpha=0.01)": Lasso(alpha=0.01, max_iter=20000),
        "Elastic Net": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=20000),
        "Random Forest": RandomForestRegressor(n_estimators=150, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=150, learning_rate=0.05, max_depth=2, random_state=RANDOM_STATE),
    }
    cv = TimeSeriesSplit(n_splits=4)
    rows = []
    for name, estimator in shortlist.items():
        pipe = Pipeline([("preprocess", clone(preprocessor)), ("model", estimator)])
        scores = cross_validate(
            pipe, X_train, y_train, cv=cv,
            scoring={"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"},
            n_jobs=-1,
            return_train_score=True
        )
        rows.append({
            "Model": name,
            "CV validation RMSE mean": -scores["test_rmse"].mean(),
            "CV validation RMSE std": scores["test_rmse"].std(),
            "CV validation MAE mean": -scores["test_mae"].mean(),
            "CV validation R2 mean": scores["test_r2"].mean(),
            "CV training RMSE mean": -scores["train_rmse"].mean()
        })
    result = pd.DataFrame(rows).sort_values("CV validation RMSE mean").reset_index(drop=True)
    print(dataset_name)
    display(result.round(3))
    return result

day_cv_results = time_cv_shortlist(day_X_train, day_y_train, day_preprocessor, "Daily time-series cross-validation")
hour_cv_results = time_cv_shortlist(hour_X_train, hour_y_train, hour_preprocessor, "Hourly time-series cross-validation")

## 16. Final scorecard and model choice

Use the validation results and diagnostics together, not only one metric. For a demand-planning use case, MAE communicates typical error in number of rentals, while RMSE gives more weight to large misses. Inspect performance over peak periods and across time.

The selected model in this notebook is the lowest-RMSE candidate from the initial model comparison, for demonstration. In a production workflow, choose based on training-period time-series validation, refit the selected pipeline on all pre-holdout data, and evaluate once on the final holdout.

In [ ]:
scorecard = pd.DataFrame([
    {
        "Dataset": "Daily",
        "Selected candidate (holdout RMSE)": day_best_name,
        "Holdout MAE": mean_absolute_error(day_y_test, day_best_pred),
        "Holdout RMSE": mean_squared_error(day_y_test, day_best_pred) ** 0.5,
        "Holdout R2": r2_score(day_y_test, day_best_pred),
        "Train rows": len(day_X_train),
        "Holdout rows": len(day_X_test),
    },
    {
        "Dataset": "Hourly",
        "Selected candidate (holdout RMSE)": hour_best_name,
        "Holdout MAE": mean_absolute_error(hour_y_test, hour_best_pred),
        "Holdout RMSE": mean_squared_error(hour_y_test, hour_best_pred) ** 0.5,
        "Holdout R2": r2_score(hour_y_test, hour_best_pred),
        "Train rows": len(hour_X_train),
        "Holdout rows": len(hour_X_test),
    }
])
display(scorecard.round(3))

## 17. Save the final pipelines

The saved pipeline includes preprocessing and the estimator, so the same transformations are applied later. The files are saved to the notebook's current working directory. Keep a record of the dataset version, feature definitions, cutoff date, and metrics alongside the model files.

In [ ]:
import joblib

MODEL_DIR = Path("saved_models")
MODEL_DIR.mkdir(exist_ok=True)

day_model_path = MODEL_DIR / "bike_daily_demand_model.joblib"
hour_model_path = MODEL_DIR / "bike_hourly_demand_model.joblib"

joblib.dump(day_best_model, day_model_path)
joblib.dump(hour_best_model, hour_model_path)

print("Saved:", day_model_path.resolve())
print("Saved:", hour_model_path.resolve())

## 18. Predict on new records (template)

New records must contain the same original predictor columns used during training, excluding `cnt`, `casual`, `registered`, and `instant`. Supply a valid `dteday` and, for hourly prediction, `hr`. The helper recreates the same engineered features and columns before prediction.

For real operations, validate allowed category values and ensure the input features are genuinely available at the moment a forecast is made.

In [ ]:
def make_new_prediction_input(new_rows, dataset_kind, fitted_pipeline, reference_X):
    # new_rows is a DataFrame with original columns such as dteday, season, yr,
    # mnth, holiday, weekday, workingday, weathersit, temp, atemp, hum, windspeed;
    # include hr for hourly prediction. cnt/casual/registered/instant are not needed.
    X_new, _, _, _, _, _ = prepare_features(new_rows.assign(cnt=0), dataset_kind)
    # Keep the time trend anchored to the original training dataset, not the new batch.
    source_dates = parse_date_column(new_rows)["dteday"]
    historical_start = day_model_data["dteday"].min() if dataset_kind == "day" else hour_model_data["dteday"].min()
    X_new["year_from_start"] = (source_dates.reset_index(drop=True) - historical_start).dt.days / 365.25
    # Align to the feature columns expected by the trained preprocessing pipeline.
    X_new = X_new.reindex(columns=reference_X.columns, fill_value=np.nan)
    return fitted_pipeline.predict(X_new)

# Example template — uncomment and fill with a realistic future record:
# daily_example = pd.DataFrame([{
#     "dteday": "2026-10-27", "season": 4, "yr": 1, "mnth": 10,
#     "holiday": 0, "weekday": 1, "workingday": 1, "weathersit": 1,
#     "temp": 0.50, "atemp": 0.48, "hum": 0.60, "windspeed": 0.15
# }])
# print(make_new_prediction_input(daily_example, "day", day_best_model, day_X_train))
#
# Hourly example needs the 'hr' field as well.

## 19. Project conclusion checklist

Before presenting the project:
- [ ] Confirm both CSV files load and date parsing is correct.
- [ ] Explain why `casual` and `registered` were excluded (target leakage).
- [ ] Compare the daily and hourly results separately.
- [ ] Report MAE, RMSE, and R² on the chronological holdout.
- [ ] Show actual-vs-predicted and residual plots.
- [ ] Discuss any residual patterns, high-demand errors, and time drift.
- [ ] Compare train vs test errors and learning curves for overfitting.
- [ ] Use time-aware validation to support model selection.
- [ ] Save the selected full pipeline and document the cutoff date.

### Important limitation
A model trained on this historical dataset may not generalize to a different city, fleet, pricing structure, or changed operating conditions. The model estimates associations in the historical data; it does not prove that changing a feature will cause demand to change.